In [ ]:
# W1. Inspect existing implementations safely; do not print credentials or response payloads.
from pathlib import Path
import json, ast, time, hashlib
from IPython.display import display, Markdown
assert 'D_PIPE' in globals() and 'D_CLEAN' in globals()
display(Markdown('# Jev world-knowledge feedback\nSame original seed-123 hand. Joint architectural interventions, matched probes, visual observations and full decision history. The earlier vector-attribution results are motivation, not a quality target.'))
W_NB=json.loads(Path('Jev_Attention_Downsampling_Hands.ipynb').read_text())
for ci,c in enumerate(W_NB['cells']):
 s=''.join(c.get('source',[]))
 if c.get('cell_type')!='code': continue
 try:
  tree=ast.parse(s)
 except SyntaxError: continue
 defs=[n.name for n in ast.walk(tree) if isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef,ast.ClassDef))]
 if any(k in s.lower() for k in ['typesafe','type-safe','qwen','requests.post','api.types']):
  print('CELL',ci, s.splitlines()[0][:100], 'DEFINITIONS', defs)
print('Loaded callable names:',[k for k,v in list(globals().items()) if callable(v) and any(x in k.lower() for x in ['jev','qwen','review'])])

In [ ]:
# W2. Read only the relevant source definitions, never the environment file.
import re
for ci,names in [(60,{'v_call','v_choice'}),(62,{'v_vision'}),(69,{'v_vision'})]:
 s=''.join(W_NB['cells'][ci]['source'])
 for n in ast.parse(s).body:
  if isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef)) and n.name in names:
   print('\nCELL',ci,ast.get_source_segment(s,n))
for name in ['Hands_Four_Methods_Computation_Dry_Run.ipynb']:
 nb=json.loads(Path(name).read_text())
 for c in nb['cells']:
  s=''.join(c.get('source',[]))
  if c.get('cell_type')!='code': continue
  try: tree=ast.parse(s)
  except SyntaxError: continue
  for n in tree.body:
   if isinstance(n,(ast.FunctionDef,ast.ClassDef)) and n.name in ['d_jev_review','d_predict','DProcessor']:
    print(ast.get_source_segment(s,n))
print('MODEL/cache directories:',[str(p) for p in Path('models').iterdir()])

In [ ]:
# W3. Inspect supported question types and the visual observer before designing prompts.
for ci in [6,60]:
 s=''.join(W_NB['cells'][ci]['source'])
 for n in ast.parse(s).body:
  if isinstance(n,(ast.Assign,ast.AnnAssign)):
   text=ast.get_source_segment(s,n)
   if ('V_' in text[:80] or 'Qwen' in text or 'from_pretrained' in text) and not any(x in text.lower() for x in ['api_key','authorization','environ','dotenv']): print(text[:10000])
for c in W_NB['cells']:
 s=''.join(c.get('source',[]))
 if c.get('cell_type')!='code':continue
 try: tree=ast.parse(s)
 except SyntaxError:continue
 for n in tree.body:
  if isinstance(n,(ast.FunctionDef,ast.ClassDef)) and n.name in ['vector_observe','vector_api']:
   print(ast.get_source_segment(s,n))
print('Dry questions:',D_JEV_QUESTIONS)
print('Observer-related live names:',[k for k in globals() if any(x in k.lower() for x in ['vlm','observer','processor','vision'])])

In [ ]:
# W4. Inspect model instrumentation and reviewer cache availability.
for c in json.loads(Path('Hands_Four_Methods_Computation_Dry_Run.ipynb').read_text())['cells']:
 s=''.join(c.get('source',[]))
 if c.get('cell_type')!='code':continue
 try:tree=ast.parse(s)
 except SyntaxError:continue
 for n in tree.body:
  if isinstance(n,(ast.ClassDef,ast.FunctionDef)) and n.name in ['DTraceProcessor','d_install','d_properties','d_sg','d_freeu_trace']:
   print(ast.get_source_segment(s,n))
  elif isinstance(n,ast.ClassDef):print('CLASS',n.name,ast.get_source_segment(s,n))
from huggingface_hub import snapshot_download
W_VLM_PATH=snapshot_download('Qwen/Qwen2-VL-2B-Instruct',local_files_only=True)
print('Qwen weights present in local cache:',bool(W_VLM_PATH))
print('Pipeline:',D_PIPE.config._name_or_path,'FreeU defaults:',D_MANIFEST['FreeU'])

In [ ]:
# W5. Frozen protocol and verbatim prompts; Jev uses the existing typed-choice API.
import copy, math, io, base64, requests, numpy as np, torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
W_OUT=Path('results')/time.strftime('jev_world_feedback_%Y%m%d_%H%M%S');W_OUT.mkdir(parents=True)
W_SEEDS=list(range(123200100,123200108))
W_PROTOCOL={'source':str(D_SOURCE_PATHS[0]),'source_sha256':hashlib.sha256(D_SOURCE.tobytes()).hexdigest(),'seeds':W_SEEDS,'schedule_steps':100,'continuation_indices':[50,99],'rounds_per_seed':10,'advance_per_round':5,'probe_steps':2,'candidates':['current configuration','Jev joint configuration','halfway from current to Jev configuration'],'Jev_calls_per_round':2,'max_Jev_calls':160,'model':'jev-1.13.0','future_endpoint_visible':False,'quality_evaluator':'Qwen2-VL-2B descriptions plus human-visible images; uncalibrated, not a quality ground truth','controls':'conditional self-attention head gates and temperature at 8x8/16x16; headwise identity perturbation guidance; FreeU; CFG; bounded hand-token area guidance; model-derived soft commit mask','scope_limit':'No learned anatomy detector, new model training, or high-resolution crop diffusion in this experiment.'}
W_WORLD='''Use your knowledge of anatomy, perspective, occlusion, materials, and image formation to explain the supplied observations. Distinguish natural overlap, foreshortening, and hidden surfaces from a malformed connection. Five visible fingers is not a universal requirement. Decide which relationship needs a coordinated change and which surrounding relationships should remain stable. The supplied visual observer can misdescribe the image: its text is an observation claim, not anatomical ground truth. Internal attention and feature measurements do not identify fingers. Use measured intervention responses to connect a desired visible change to an architectural experiment. A promising numerical direction is not automatically a visual improvement. Consider competing explanations. Choose the supplied hypothesis or unresolved option that best fits the actual evidence, and choose joint bounded controls that test a useful consequence. Do not invent observations or tensor-group anatomy.'''
W_PROPOSE='''The state includes the fixed original image request, current visual descriptions, recent signed property errors, actual attention transformations, per-head response statistics, collateral-change measurements and the history of tested joint configurations. Each question in this request is evaluated independently. Do not assume access to another answer in this same request. Select an ABSOLUTE parameter setting for the next joint probe, using the same supplied state for every component. The executor combines all answers, tests that joint configuration and its halfway-strength counterpart from an identical checkpoint, then asks for a selection. No scalar is an anatomy label. Holding current settings is allowed; uncertain but discriminating bounded probes are also allowed. The outer loop always completes all fifty denoising steps. The model does not choose whether to stop.'''
W_SELECT='''The candidate records were generated from the same checkpoint, with the same prompt, noise schedule and number of denoising advances. They include independent visual descriptions, matched image differences, signed attention-property errors, per-head statistics, and exact applied controls. Use world knowledge to assess connections, articulation, occlusion, proportions, lighting and material continuity while preserving pose and surrounding content. A stronger edge, lower attention loss, greater stability or larger change alone does not establish a better hand. Choose the candidate with the most supported useful consequence, or the unchanged configuration if indistinguishable. Evaluate collateral effects and unresolved risk. Short probes can be inconclusive; preserve that uncertainty in your answers. No candidate includes a future completed endpoint. Candidate IDs and presentation order are shuffled independently of method. The retained branch will advance three further steps and be reviewed in the next round.'''
W_OBSERVER='''Describe only what is visible in this image of a hand. Report the visible pose, distinct contours, finger-to-palm connections, relative proportions, joint bends, overlaps, shading, and surface detail. Identify concrete suspicious relationships and their approximate position, distinguishing them from plausible perspective or occlusion. State what cannot be assessed. Do not infer the generation method, count hidden fingers, prescribe tensor edits, or treat five fingers/sharpness alone as correctness. Be concise and separate observations from uncertainty.'''
W_HYPOTHESES={'occlusion':'Contours may be hidden by a plausible overlap or perspective; preserve geometry while checking continuity.','connection':'Visible contour or attachment relationships may be inconsistent with connected fingers and palm.','proportion':'Relative lengths or widths may be implausible after allowing for perspective.','articulation':'Joint bends or orientation relationships may be inconsistent with a plausible pose.','surface':'Large structure seems plausible; boundary, lighting or material continuity may need refinement.','global':'Pose, composition or surrounding content may be drifting from the request.','unresolved':'No supported defect or insufficient evidence to identify the relationship.'}
def w_dump(name,obj):
 text=json.dumps(obj,indent=2,default=lambda x:x.tolist() if hasattr(x,'tolist') else str(x))
 assert _d_key not in text
 (W_OUT/name).write_text(text)
w_dump('protocol.json',W_PROTOCOL);w_dump('verbatim_prompts.json',{'world':W_WORLD,'proposal':W_PROPOSE,'selection':W_SELECT,'observer':W_OBSERVER,'hypotheses':W_HYPOTHESES})
D_SOURCE.save(W_OUT/'original_seed123.png')
for label,prompt in [('World knowledge',W_WORLD),('Joint proposal',W_PROPOSE),('Branch selection',W_SELECT),('Visual observer',W_OBSERVER)]:
 display(Markdown('### '+label+'\n'+prompt))
print('Output:',W_OUT,'Frozen continuation seeds:',W_SEEDS)


In [ ]:
# W6. Reuse cached Qwen weights; review pixels without seeing intervention labels.
from transformers import AutoProcessor,Qwen2VLForConditionalGeneration
W_VP=AutoProcessor.from_pretrained(W_VLM_PATH,min_pixels=64*28*28,max_pixels=256*28*28,local_files_only=True)
W_VM=Qwen2VLForConditionalGeneration.from_pretrained(W_VLM_PATH,torch_dtype=torch.bfloat16,use_safetensors=True,attn_implementation='sdpa',local_files_only=True).eval().to('cuda')
W_VIS_CACHE={};W_VIS_SECONDS=0.
@torch.inference_mode()
def w_observe(images):
 global W_VIS_SECONDS
 reports=[]
 for im in images:
  key=hashlib.sha256(im.tobytes()+W_OBSERVER.encode()).hexdigest()
  if key not in W_VIS_CACHE:
   t=time.time();messages=[{'role':'user','content':[{'type':'image'},{'type':'text','text':W_OBSERVER}]}]
   template=W_VP.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
   inp=W_VP(text=[template],images=[im],padding=True,return_tensors='pt').to('cuda')
   out=W_VM.generate(**inp,max_new_tokens=220,do_sample=False)
   W_VIS_CACHE[key]=W_VP.batch_decode(out[:,inp.input_ids.shape[1]:],skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]
   W_VIS_SECONDS+=time.time()-t
  reports.append({'image_sha256':key,'report':W_VIS_CACHE[key],'source':'Qwen2-VL-2B, uncalibrated visual description'})
 return reports
W_SOURCE_REVIEW=w_observe([D_SOURCE])[0]
w_dump('source_review.json',W_SOURCE_REVIEW)
display(D_SOURCE);print(W_SOURCE_REVIEW['report'])
print('GPU free GiB:',round(torch.cuda.mem_get_info()[0]/2**30,2))

In [ ]:
# W7. Exact editable attention transformations and multilevel measurements.
W_LAYERS={'8x8':D_SELF,'16x16':D_CROSS.replace('attn2','attn1')}
W_NEUTRAL={'layer':'8x8','gates':[1.]*8,'u':[0.]*8,'temperature':1.,'cfg':7.5,'freeu':0.,'sg_area':1.,'sg_gain':0.,'mask':'full'}
W_COUNTS={'full_forwards':0,'partial_gradient_forwards':0,'backwards':0,'jev_calls':0,'jev_seconds':0.}
W_ACTIVE={};W_TRACE={}
class WAttention:
 def __init__(self,name):self.name=name
 def __call__(self,attn,hidden_states,encoder_hidden_states=None,attention_mask=None,temb=None,*args,**kwargs):
  residual=hidden_states;ndim=hidden_states.ndim
  if attn.spatial_norm is not None:hidden_states=attn.spatial_norm(hidden_states,temb)
  if ndim==4:
   b,ch,hh,ww=hidden_states.shape;hidden_states=hidden_states.view(b,ch,hh*ww).transpose(1,2)
  b,n,ch=hidden_states.shape
  if attn.group_norm is not None:hidden_states=attn.group_norm(hidden_states.transpose(1,2)).transpose(1,2)
  enc=hidden_states if encoder_hidden_states is None else encoder_hidden_states
  if encoder_hidden_states is not None and attn.norm_cross:enc=attn.norm_encoder_hidden_states(enc)
  h=attn.heads;d=attn.inner_dim//h
  q=attn.to_q(hidden_states).view(b,n,h,d).transpose(1,2)
  k=attn.to_k(enc).view(b,-1,h,d).transpose(1,2);v=attn.to_v(enc).view(b,-1,h,d).transpose(1,2)
  assert attention_mask is None
  out=F.scaled_dot_product_attention(q,k,v,dropout_p=0.)
  if self.name==D_CROSS:
   prob=(q[-1].float()@k[-1].float().transpose(-1,-2)*attn.scale).softmax(-1)
   W_TRACE['cross_maps']=prob.mean(0).transpose(0,1).reshape(77,16,16)
  if self.name in W_LAYERS.values():
   logits=q[-1].float()@k[-1].float().transpose(-1,-2)*attn.scale
   a=logits.softmax(-1);selected=self.name==W_LAYERS[W_ACTIVE['layer']]
   temp=W_ACTIVE['temperature'] if selected else 1.
   a_edit=(logits/temp).softmax(-1)
   u=torch.tensor(W_ACTIVE['u'] if selected and W_ACTIVE.get('perturb') else [0.]*h,device=q.device)[:,None,None]
   a_edit=(1-u)*a_edit+u*torch.eye(n,device=q.device)[None]
   gates=torch.tensor(W_ACTIVE['gates'] if selected else [1.]*h,device=q.device)[:,None,None]
   hbase=a@v[-1].float();hedit=(a_edit@v[-1].float())*gates
   out=out.clone();out[-1]=hedit.to(out.dtype)
   diff=hedit-hbase
   W_TRACE[self.name]={'grid':int(n**.5),'entropy':(-(a*a.clamp_min(1e-12).log()).sum(-1).mean(-1)).detach().cpu().tolist(),'attention_L1_change':(a_edit-a).abs().mean((1,2)).detach().cpu().tolist(),'head_output_rms':hbase.square().mean((1,2)).sqrt().detach().cpu().tolist(),'head_delta_rms':diff.square().mean((1,2)).sqrt().detach().cpu().tolist(),'identity_minus_attention_output_rms':(v[-1].float()-hbase).square().mean((1,2)).sqrt().detach().cpu().tolist(),'q_rms':q[-1].float().square().mean((1,2)).sqrt().cpu().tolist(),'k_rms':k[-1].float().square().mean((1,2)).sqrt().cpu().tolist(),'attention_pool_4x4':F.adaptive_avg_pool2d(a,(4,4)).cpu().tolist(),'head_delta_pool':F.adaptive_avg_pool2d(diff.square().mean(-1).sqrt().reshape(h,1,int(n**.5),int(n**.5)),(4,4)).cpu().tolist()}
  out=out.transpose(1,2).reshape(b,n,h*d).to(q.dtype);out=attn.to_out[1](attn.to_out[0](out))
  if ndim==4:out=out.transpose(-1,-2).reshape(b,ch,hh,ww)
  if attn.residual_connection:out=out+residual
  return out/attn.rescale_output_factor
@torch.no_grad()
def w_forward(z,i,p,perturb=False):
 global W_ACTIVE,W_TRACE
 W_ACTIVE=dict(p,perturb=perturb);W_TRACE={};W_COUNTS['full_forwards']+=1
 D_PIPE.unet.set_attn_processor({n:WAttention(n) if n in [*W_LAYERS.values(),D_CROSS] else v for n,v in D_ORIGINAL.items()})
 amount=p['freeu'];fu={k:1+amount*(v-1) for k,v in D_MANIFEST['FreeU'].items()}
 try:
  if amount:D_PIPE.unet.enable_freeu(**fu)
  pair=D_PIPE.unet(z.repeat(2,1,1,1),D_SCHED.timesteps[i],encoder_hidden_states=D_EMB).sample.chunk(2)
  trace=W_TRACE
  return pair,trace
 finally:
  D_PIPE.unet.disable_freeu();D_PIPE.unet.set_attn_processor(dict(D_ORIGINAL))
def w_eval(z,i,p,target_anchor):
 with torch.no_grad():
  bp,bt=w_forward(z,i,W_NEUTRAL);base=d_cfg(bp)
  if p==W_NEUTRAL:pair,tr=bp,bt
  else:pair,tr=w_forward(z,i,p)
  eps=pair[0]+p['cfg']*(pair[1]-pair[0])
  if any(p['u']):
   pert,_=w_forward(z,i,p,True);eps=eps+pair[1]-pert[1]
  prop,mask=d_properties(tr['cross_maps'][D_HAND].mean(0));target=target_anchor.clone();target[0]*=p['sg_area']
  info={'index':i,'properties':prop.cpu().tolist(),'target':target.cpu().tolist(),'signed_error':(prop-target).cpu().tolist(),'heads':{k:tr[v] for k,v in W_LAYERS.items()},'freeu_factors':{k:1+p['freeu']*(v-1) for k,v in D_MANIFEST['FreeU'].items()}}
 if p['sg_gain']>0:
  g,sg=d_sg(z,i,target);W_COUNTS['partial_gradient_forwards']+=1;W_COUNTS['backwards']+=1
  raw=p['sg_gain']*g;cap=.03*d_rms(base);factor=min(1.,cap/max(d_rms(raw),1e-12))
  eps=(eps.float()+raw*factor).half();info['sg']={k:v for k,v in sg.items() if k not in ['map','mask']};info['sg_cap_factor']=factor
 with torch.no_grad():
  sm=F.interpolate(mask[None,None],size=z.shape[-2:],mode='bilinear',align_corners=False).clamp(0,1)
  delta=eps-base
  if p['mask']=='hand_token':eps=base+sm*delta
  direct=p_clean(z,eps,i)-p_clean(z,base,i)
  info.update(delta_eps_rms=d_rms(eps-base),clean_effect_rms=d_rms(direct),inside_effect_rms=d_rms(direct*sm),outside_effect_rms=d_rms(direct*(1-sm)),mask_note='Soft hand-token attention proxy, not anatomical segmentation')
 return eps.detach(),p_clean(z,eps,i).detach(),info,sm.detach()
def w_start(seed):
 noise=torch.randn(D_CLEAN.shape,generator=torch.Generator(device='cuda').manual_seed(seed),device='cuda',dtype=D_CLEAN.dtype)
 return D_SCHED.add_noise(D_CLEAN,noise,D_SCHED.timesteps[50:51])
print('Attention: A=softmax(QK^T/sqrt(d)/temperature); conditional head output = gate * A V; perturbation uses A_p=(1-u)A+uI. PAG correction = eps_cond - eps_cond_perturbed.')
print('Layer catalogue:',W_LAYERS)


In [ ]:
# W8. Numerical preflight: neutral equivalence, repeatability and one joint edit.
_wz=w_start(W_SEEDS[0])
with torch.no_grad():
 _wp,_wt=w_forward(_wz,50,W_NEUTRAL)
 W_ANCHOR,_=d_properties(_wt['cross_maps'][D_HAND].mean(0));W_ANCHOR=W_ANCHOR.detach()
 _old,_=d_predict(_wz,50)
 _neutral_error=d_rms(d_cfg(_wp)-d_cfg(_old))
_we,_wc,_wi,_wm=w_eval(_wz,50,W_NEUTRAL,W_ANCHOR)
_we2,_,_,_=w_eval(_wz,50,W_NEUTRAL,W_ANCHOR)
_test=copy.deepcopy(W_NEUTRAL);_test.update(layer='16x16',temperature=.95,freeu=.25,sg_area=1.05,sg_gain=.25,mask='hand_token');_test['gates'][0]=1.05;_test['u'][1]=.25
_ee,_ec,_ei,_em=w_eval(_wz,50,_test,W_ANCHOR)
W_PREFLIGHT={'neutral_vs_previous_epsilon_rms':_neutral_error,'repeat_max_abs':float((_we-_we2).abs().max()),'joint_effect_rms':d_rms(_ee-_we),'finite':bool(torch.isfinite(_ee).all()),'processors_restored':all(D_PIPE.unet.attn_processors[k] is v for k,v in D_ORIGINAL.items()),'temporary_probe_only':True}
w_dump('preflight.json',W_PREFLIGHT);print(W_PREFLIGHT)
assert _neutral_error<.01 and W_PREFLIGHT['repeat_max_abs']==0 and W_PREFLIGHT['finite'] and W_PREFLIGHT['processors_restored']
assert W_PREFLIGHT['joint_effect_rms']>0
print('All transformations produce finite changes; original processors restored.')


In [ ]:
# W8b. Remove a neutral-path precision difference discovered by the preflight.
# Passive instrumentation must leave PyTorch's original fused attention output untouched.
_wsrc=next(s for s in get_ipython().history_manager.input_hist_raw if s.startswith('# W7. Exact editable'))
_wtree=ast.parse(_wsrc);_wclass=next(n for n in _wtree.body if isinstance(n,ast.ClassDef) and n.name=='WAttention')
_wcode=ast.get_source_segment(_wsrc,_wclass)
_wcode=_wcode.replace('hbase=a@v[-1].float();hedit=(a_edit@v[-1].float())*gates','hbase=out[-1].float();changed=selected and (temp!=1. or bool((u!=0).any()) or bool((gates!=1).any()))\n   hedit=(a_edit@v[-1].float())*gates if changed else hbase')
exec(compile(_wcode,'world_attention_neutral_fix','exec'))
(W_OUT/'attention_processor.py').write_text(_wcode)
with torch.no_grad():
 _wp,_=w_forward(_wz,50,W_NEUTRAL);_old,_=d_predict(_wz,50)
 _neutral_error=d_rms(d_cfg(_wp)-d_cfg(_old))
W_PREFLIGHT['neutral_vs_previous_epsilon_rms_after_fix']=_neutral_error
w_dump('preflight.json',W_PREFLIGHT);print('Neutral epsilon RMS after fix:',_neutral_error)
assert _neutral_error==0.,'Neutral instrumentation must exactly preserve baseline output.'


In [ ]:
# W9. Typed question catalogue and validated, archived Jev transport.
W_LEVELS={'layer':['8x8','16x16'],'temperature':[.85,1.,1.15],'cfg':[6.,7.5,9.],'freeu':[0.,.5,1.],'sg_area':[.9,1.,1.1],'sg_gain':[0.,.25,1.],'mask':['full','hand_token']}
W_CONTROL_NOTES={'layer':'Choose the editable self-attention module; no layer has an anatomical identity.','temperature':'Divide conditional attention logits by this value; below one concentrates mixing, above one spreads it.','cfg':'Unconditional + cfg * (conditional - unconditional) noise prediction.','freeu':'Interpolate the two FreeU backbone and skip factors from neutral 1 to the declared factors.','sg_area':'Target soft hand-token attention area relative to a fixed per-seed initial reference; it is not finger size.','sg_gain':'Multiply the raw gradient of the declared neutral-model property loss; cap its noise-prediction RMS contribution at 3%, never normalize small gradients upward.','mask':'Commit the change in noise prediction over all latent positions, or weight it by the soft hand-token map. This is not a segmentation or finger box.'}
W_EXPECT={'geometry':'More plausible contour/attachment/proportion relationships without pose drift.','occlusion':'More coherent contour continuation and shading at overlapping structures.','detail':'More coherent material or boundary detail while preserving geometry.','preserve':'Preserve the current hand while continuing denoising.','discriminate':'Measure whether the joint control changes the hypothesized relationship; improvement is unresolved.'}
W_COUNTER={'no_effect':'No corresponding visible relationship changes despite a numerical response.','collateral':'Desired region changes together with damage to pose, neighbors, or background.','contrary':'The visible relationship changes in the opposite or less plausible way.','observer_conflict':'Descriptions disagree, or make claims unsupported by the retained pixels.','persistence':'An early apparent benefit disappears after further denoising.'}
def w_choice(text,criteria):return {'type':'choice','instructions':text,'criteria':criteria}
def w_proposal_questions(p):
 q={'hypothesis':w_choice(W_WORLD,W_HYPOTHESES),'expected':w_choice('Use the supplied observations and intervention history to choose the plausible visible consequence of a useful new probe. This is a prediction, not an outcome.',W_EXPECT),'counterevidence':w_choice('Which subsequent observation most directly challenges the current account of the hand or control response?',W_COUNTER)}
 values={}
 for k,levels in W_LEVELS.items():
  vals=list(dict.fromkeys([p[k],*levels]));values[k]={f'v{j}':v for j,v in enumerate(vals)}
  q[k]=w_choice(W_CONTROL_NOTES[k]+' Select one absolute value as part of the joint test. Consult protocol and current configuration in state.',{j:str(v) for j,v in values[k].items()})
 for h in range(8):
  for key,levels,note in [('gates',[.9,1.,1.1],'Scale this conditional head output before the learned output projection'),('u',[0.,.25,.5,1.],'Interpolate this head attention with the identity in the auxiliary perturbation pass')]:
   name=f'{key}_{h}';vals=list(dict.fromkeys([p[key][h],*levels]));values[name]={f'v{j}':v for j,v in enumerate(vals)}
   q[name]=w_choice(f'{note}. Select absolute head-{h} setting for a joint probe. Layer choice is independent, so use responses from both offered layers and do not assume its answer. No head represents an identified finger.',{j:str(v) for j,v in values[name].items()})
 return q,values
def w_call(tag,state,questions):
 assert W_COUNTS['jev_calls']<W_PROTOCOL['max_Jev_calls']
 W_COUNTS['jev_calls']+=1;serial=W_COUNTS['jev_calls']
 payload={'model':W_PROTOCOL['model'],'state':state,'questions':questions}
 w_dump(f'api_{serial:03}_{tag}_request.json',payload);t=time.time()
 for attempt in range(3):
  r=requests.post('https://api.typesafe.ai/v1/systemone',headers={'Authorization':'Bearer '+_d_key},json=payload,timeout=90)
  if r.status_code==200:break
  w_dump(f'api_{serial:03}_http_attempt{attempt}.json',{'status':r.status_code,'seconds':time.time()-t})
  if r.status_code not in [429,500,502,503,504,529]:raise RuntimeError(f'Jev HTTP {r.status_code}; request retained; body withheld')
  time.sleep(2**attempt)
 if r.status_code!=200:raise RuntimeError('Jev transport retries exhausted')
 result=r.json();w_dump(f'api_{serial:03}_{tag}_response.json',result);W_COUNTS['jev_seconds']+=time.time()-t
 answers=result['answers'];decoded={}
 for k,q in questions.items():
  v=answers[k]['choice'];assert v in q['criteria'],f'Invalid Jev choice for {k}'
  decoded[k]=v
 return decoded
def w_decode(p,a,values):
 new=copy.deepcopy(p)
 for k in W_LEVELS:new[k]=values[k][a[k]]
 for h in range(8):
  for k in ['gates','u']:new[k][h]=values[f'{k}_{h}'][a[f'{k}_{h}']]
 return new
def w_half(old,new):
 p=copy.deepcopy(new)
 for k in ['temperature','cfg','freeu','sg_area','sg_gain']:p[k]=(old[k]+new[k])/2
 for k in ['gates','u']:p[k]=[(a+b)/2 for a,b in zip(old[k],new[k])]
 return p
print('Independent choice fields per proposal:',len(w_proposal_questions(W_NEUTRAL)[0]))
print('No unsupported free-text generation endpoint: hypotheses/predictions/falsifiers are typed choices; raw API answers are archived.')


In [ ]:
# W10. Deterministic loop, evidence ledger, matched probes and chronological viewer.
import random, html, traceback
W_DECISIONS=[];W_FINALS={};W_FAILURES=[];W_RUNNING=False
W_ARCH={'latent':'VAE scaled latent 4x64x64; clean forecasts are provisional, not completed images','conditioning':'Frozen CLIP text features 77x768; original prompt fixed','layers':W_LAYERS,'transformation':'A=softmax(QK^T/sqrt(d)/T); conditional output=g_h*A*V; auxiliary A_p=(1-u_h)*A+u_h*I; eps=eps_uncond+CFG*(eps_cond-eps_uncond)+(eps_cond-eps_perturbed)','FreeU_endpoints':D_MANIFEST['FreeU'],'SG_reference':'Raw gradient of neutral-model soft hand-token area/centroid loss, fixed per-seed anchor; cap only, no minimum imposed gradient magnitude','interpretation':'These controls are mechanisms, not anatomical identities. No future endpoint or seed-specific answer is available.'}
def w_small(info):
 keep={k:v for k,v in info.items() if k!='heads'}
 keep['heads']={layer:{k:v for k,v in rec.items() if k not in ['attention_pool_4x4','head_delta_pool']} for layer,rec in info['heads'].items()}
 return keep
def w_image_delta(im,ref,mask):
 a=np.asarray(im).astype('float32')/255;b=np.asarray(ref).astype('float32')/255
 m=F.interpolate(mask.float(),size=a.shape[:2],mode='bilinear',align_corners=False)[0,0].cpu().numpy();err=np.abs(a-b).mean(-1)
 return {'RGB_MAE':float(err.mean()),'inside_weighted_MAE':float((err*m).sum()/max(m.sum(),1e-9)),'outside_weighted_MAE':float((err*(1-m)).sum()/max((1-m).sum(),1e-9)),'identical_pixels':bool(np.array_equal(a,b))}
def w_thumb(im,width=220):
 b=io.BytesIO();im.resize((width,round(im.height*width/im.width))).save(b,format='JPEG',quality=85)
 return 'data:image/jpeg;base64,'+base64.b64encode(b.getvalue()).decode()
def w_report():
 parts=['<!doctype html><html><head><meta charset="utf-8"><meta http-equiv="refresh" content="30"><title>Jev world-knowledge feedback</title><style>body{font:15px system-ui;background:#16191d;color:#eee;margin:24px} .row{display:flex;gap:12px;flex-wrap:wrap} article{border:1px solid #555;padding:16px;margin:18px 0} figure{margin:4px} pre{white-space:pre-wrap} a{color:#9cd}</style></head><body><h1>Jev world-knowledge feedback</h1>']
 parts+=['<p>Original seed-123 source. Every attempt retained. Images after each round are clean forecasts until the final step. Visual judgments are uncalibrated. No quality or speedup claim.</p>',f'<p>Completed rounds: {len(W_DECISIONS)}/80; finished seeds: {len(W_FINALS)}/8; Jev calls: {W_COUNTS["jev_calls"]}/160; running: {W_RUNNING}</p>',f'<img width="256" src="{w_thumb(D_SOURCE,256)}"><p>Original fixed source</p>']
 for seed in W_SEEDS:
  rows=[r for r in W_DECISIONS if r['seed']==seed]
  if not rows:continue
  parts.append(f'<h2>Continuation seed {seed}</h2><div class="row">')
  for r in rows:parts.append(f'<figure><img src="{w_thumb(Image.open(W_OUT/r["committed_image"]))}"><figcaption>Round {r["round"]+1}; schedule {r["next_index"]}</figcaption></figure>')
  parts.append('</div>')
  for r in rows:
   parts.append(f'<article><h3>Round {r["round"]+1}: hypothesis {html.escape(r["proposal_answers"]["hypothesis"])}; chose {html.escape(r["selected_kind"])}</h3><div class="row">')
   for c in r['candidates']:parts.append(f'<figure><img src="{w_thumb(Image.open(W_OUT/c["image"]))}"><figcaption>{html.escape(c["kind"])} — presented as {c["id"]}</figcaption></figure>')
   parts.append('</div><details><summary>Full decision, parameter changes and observations</summary><pre>'+html.escape(json.dumps(r,indent=2))+'</pre></details></article>')
 if W_FAILURES:parts.append('<h2>Failures retained</h2><pre>'+html.escape(json.dumps(W_FAILURES,indent=2))+'</pre>')
 parts.append('</body></html>');(W_OUT/'live.html').write_text(''.join(parts))
@torch.no_grad()
def w_step(z,eps,i):return D_SCHED.step(eps,D_SCHED.timesteps[i],z,eta=0.).prev_sample

def w_run_seed(seed):
 z=w_start(seed);p=copy.deepcopy(W_NEUTRAL);history=[];signed=[]
 with torch.no_grad():
  _,t=w_forward(z,50,W_NEUTRAL);anchor,_=d_properties(t['cross_maps'][D_HAND].mean(0));anchor=anchor.detach()
 for round_id,i in enumerate(range(50,100,5)):
  tag=f's{seed}_r{round_id:02}'
  e,clean,info,mask=w_eval(z,i,p,anchor);current_image=d_decode(clean)
  current_image.save(W_OUT/f'{tag}_before.png')
  visual=w_observe([current_image])[0]
  state={'protocol':W_PROPOSE,'world_knowledge':W_WORLD,'architecture':W_ARCH,'original_prompt':D_PROMPT,'negative_prompt':D_NEGATIVE if 'D_NEGATIVE' in globals() else 'Original negative prompt unchanged','original_source_review':W_SOURCE_REVIEW,'schedule_index':i,'current_configuration':p,'current_visual':visual,'current_internal':w_small(info),'recent_signed_errors':signed[-5:],'past_interventions':history[-3:],'observer_warning':'Earlier Qwen comparisons hallucinated differences for identical pixels; descriptions and quality judgments remain uncalibrated.'}
  qs,vals=w_proposal_questions(p);answers=w_call(tag+'_proposal',state,qs);proposed=w_decode(p,answers,vals)
  configs=[('continue',copy.deepcopy(p)),('joint',proposed),('halfway',w_half(p,proposed))]
  rng=random.Random(seed+round_id);rng.shuffle(configs);candidates=[];cache={}
  torch.save({'latent':z.cpu(),'index':i,'current_params':p,'anchor':anchor.cpu()},W_OUT/f'{tag}_checkpoint.pt')
  for j,(kind,cp) in enumerate(configs):
   key=json.dumps(cp,sort_keys=True)
   if key in cache:
    zz,pc,rows=cache[key];zz=zz.clone();pc=pc.clone();rows=copy.deepcopy(rows)
   else:
    zz=z.clone();rows=[]
    for k in range(i,i+2):
     pe,pc,pi,pm=w_eval(zz,k,cp,anchor);rows.append(pi);zz=w_step(zz,pe,k)
    cache[key]=(zz.clone(),pc.clone(),copy.deepcopy(rows))
   im=d_decode(pc);filename=f'{tag}_probe{j}.png';im.save(W_OUT/filename)
   torch.save({'post_probe_latent':zz.cpu(),'clean_forecast':pc.cpu(),'parameters':cp,'measurements':rows},W_OUT/f'{tag}_probe{j}.pt')
   candidates.append({'id':f'candidate_{j}','kind':kind,'parameters':cp,'image':filename,'im':im,'z':zz,'rows':rows})
  ref=next(c for c in candidates if c['kind']=='continue');reviews=w_observe([c['im'] for c in candidates])
  records=[]
  for c,review in zip(candidates,reviews):
   record={k:c[k] for k in ['id','kind','parameters','image']};record.update(visual=review,measurements=[w_small(x) for x in c['rows']],matched_change=w_image_delta(c['im'],ref['im'],mask));records.append(record)
  selection_state={'world_knowledge':W_WORLD,'protocol':W_SELECT,'original_prompt':D_PROMPT,'current_visual':visual,'hypothesis':W_HYPOTHESES[answers['hypothesis']],'predicted_consequence':W_EXPECT[answers['expected']],'challenging_observation':W_COUNTER[answers['counterevidence']],'candidates':[{k:v for k,v in rec.items() if k not in ['kind','image']} for rec in records],'current_configuration_candidate':ref['id'],'recent_history':history[-2:]}
  sq={'candidate':w_choice(W_SELECT,{c['id']:'Retain the candidate with this ID; inspect its supplied evidence.' for c in candidates}),'prediction_result':w_choice('Compare the declared prediction to the actually supplied matched outcomes.',{'supported':'A specific predicted visible relationship changed in the expected direction.','contradicted':'Observed changes oppose the prediction or damage preserved relationships.','unresolved':'Evidence is insufficient, invisible, contradictory or no discernible change.'}),'risk':w_choice('Which uncertainty should the next round inspect?',W_COUNTER),'quality':w_choice('What relative image-quality claim is supported by these short matched probes?',{'better':'Specific improvement with no material collateral change is supported by descriptions.','worse':'Specific degradation is supported by descriptions.','unresolved':'Quality is unresolved; numerical effect or generic praise is insufficient.'})}
  choice=w_call(tag+'_selection',selection_state,sq)
  chosen=next(c for c in candidates if c['id']==choice['candidate']);z=chosen['z'];p=copy.deepcopy(chosen['parameters'])
  signed.extend([x['signed_error'] for x in chosen['rows']])
  for k in range(i+2,i+5):
   ee,cc,ii,mm=w_eval(z,k,p,anchor);signed.append(ii['signed_error']);z=w_step(z,ee,k)
   torch.save({'index':k,'latent_after':z.cpu(),'clean_forecast':cc.cpu(),'measurements':ii,'parameters':p},W_OUT/f'{tag}_committed_step{k}.pt')
  final_im=d_decode(z if i+5==100 else cc);fname=f'{tag}_committed.png';final_im.save(W_OUT/fname)
  record={'seed':seed,'round':round_id,'index':i,'next_index':i+5,'before_image':f'{tag}_before.png','proposal_answers':answers,'proposed_configuration':proposed,'selection_answers':choice,'selected_kind':chosen['kind'],'selected_id':chosen['id'],'committed_parameters':p,'committed_image':fname,'candidates':records,'counts':copy.deepcopy(W_COUNTS)}
  W_DECISIONS.append(record);w_dump(f'{tag}_decision.json',record)
  history.append({'index':i,'hypothesis':answers['hypothesis'],'predicted':answers['expected'],'counterevidence':answers['counterevidence'],'selected':chosen['kind'],'parameters':p,'matched_change':next(r['matched_change'] for r in records if r['id']==chosen['id']),'review':choice})
  w_report();print(f'Seed {seed}, round {round_id+1}/10: hypothesis={answers["hypothesis"]}; selected={chosen["kind"]}; probe quality={choice["quality"]}',flush=True)
  display(final_im.resize((320,320)))
 W_FINALS[seed]=fname;w_dump(f's{seed}_final_review.json',w_observe([final_im])[0]);w_dump('final_images.json',W_FINALS)
 return fname
w_report();display(HTML(f'<a target="_blank" href="/files/workspace/crazy_exp/{W_OUT}/live.html">Open live progression: every round, every probe, every decision</a>'))
print('Ready. The executor never changes the source or skips a scheduled denoising advance.')


In [ ]:
# W11. Run the frozen eight-seed experiment; retain failures and never choose a final attempt by appearance.
assert W_PREFLIGHT['neutral_vs_previous_epsilon_rms_after_fix']==0
assert not W_RUNNING and not W_DECISIONS,'Do not launch this cell twice; inspect saved checkpoints before any recovery.'
# Archive executable cell source, including the original failed precision check and its correction.
_wcells=[s for s in get_ipython().history_manager.input_hist_raw if re.match(r'^# W\d+',s)]
(W_OUT/'executed_cells.py').write_text('\n\n'.join(_wcells))
W_RUNNING=True;W_RUN_START=time.time();w_report()
try:
 for seed in W_SEEDS:
  print('Starting continuation seed',seed,flush=True)
  w_run_seed(seed)
 print('COMPLETE:',len(W_FINALS),'seeds;',len(W_DECISIONS),'rounds;',W_COUNTS['jev_calls'],'Jev calls.',flush=True)
except Exception as exc:
 failure={'type':type(exc).__name__,'message':str(exc),'traceback':traceback.format_exc(),'completed_rounds':len(W_DECISIONS),'completed_seeds':list(W_FINALS)}
 W_FAILURES.append(failure);w_dump('failure.json',failure)
 print('RUN PAUSED ON ERROR:',type(exc).__name__,str(exc),flush=True)
 raise
finally:
 W_RUNNING=False
 w_dump('run_status.json',{'running':False,'complete':len(W_FINALS)==8,'completed_rounds':len(W_DECISIONS),'counts':W_COUNTS,'wall_seconds':time.time()-W_RUN_START,'visual_seconds':W_VIS_SECONDS})
 w_report()


In [ ]:
# W12. Audit the completed run: actual decisions, effective edits, and retained evidence.
from collections import Counter
import pandas as pd
print('STATUS',json.loads((W_OUT/'run_status.json').read_text()))
print('FAILURES',W_FAILURES)
print('SELECTIONS',dict(Counter(r['selected_kind'] for r in W_DECISIONS)))
print('HYPOTHESES',dict(Counter(r['proposal_answers']['hypothesis'] for r in W_DECISIONS)))
print('QUALITY',dict(Counter(r['selection_answers']['quality'] for r in W_DECISIONS)))
W_AUDIT=[]
for r in W_DECISIONS:
 current=next(c for c in r['candidates'] if c['kind']=='continue')
 joint=next(c for c in r['candidates'] if c['kind']=='joint')
 changed={k:{'from':current['parameters'][k],'to':v} for k,v in joint['parameters'].items() if v!=current['parameters'][k]}
 W_AUDIT.append({'seed':r['seed'],'round':r['round']+1,'selected':r['selected_kind'],'changed_fields':list(changed),'change_detail':changed,'joint_RGB_MAE':joint['matched_change']['RGB_MAE'],'identical_joint':joint['matched_change']['identical_pixels']})
W_ADF=pd.DataFrame(W_AUDIT)
print('PROPOSAL changed-field counts',dict(Counter(k for a in W_AUDIT for k in a['changed_fields'])))
print('PROPOSALS with any changed field',sum(bool(a['changed_fields']) for a in W_AUDIT),'/',len(W_AUDIT))
print('PROBES with any RGB change',sum(not a['identical_joint'] for a in W_AUDIT))
print('Joint RGB change range',W_ADF['joint_RGB_MAE'].min(),W_ADF['joint_RGB_MAE'].max())
print('EXAMPLES of changed proposals',[a for a in W_AUDIT if a['changed_fields']][:4])
print('ORIGINAL REVIEW:',W_SOURCE_REVIEW)
print('FINAL REVIEWS:')
for seed in W_SEEDS:
 f=W_OUT/f's{seed}_final_review.json'
 if f.exists():print(seed,json.loads(f.read_text())['report'])
w_dump('audit_decisions.json',W_AUDIT)


In [ ]:
# W13. Inspect every final image and test whether retained generation was ordinary diffusion.
assert len(W_DECISIONS)==80 and len(W_FINALS)==8
W_ALL_NEUTRAL=all(r['committed_parameters']==W_NEUTRAL for r in W_DECISIONS)
print('Every retained configuration exactly neutral:',W_ALL_NEUTRAL)
print('Unique joint parameter vectors:',len({json.dumps(r['proposed_configuration'],sort_keys=True) for r in W_DECISIONS}))
for field in ['layer','temperature','cfg','freeu','sg_area','sg_gain','mask','gates','u']:
 print(field,dict(Counter(json.dumps(r['proposed_configuration'][field]) for r in W_DECISIONS)))
W_AUDIT_HTML='''<!doctype html><html><head><meta charset="utf-8"><title>Completed Jev run audit</title><style>body{font:16px system-ui;background:#171a20;color:#eee;margin:20px}.grid{display:grid;grid-template-columns:repeat(3,minmax(220px,1fr));gap:16px}img{width:100%}figure{margin:0}figcaption{padding:8px}a{color:#adf}pre{white-space:pre-wrap}</style></head><body><h1>Completed run: all eight outcomes</h1><p>80/80 choices retained ordinary continuation. No Jev intervention was committed. These images show seed-dependent ordinary refinement, not demonstrated Jev improvement.</p><div class="grid">'''
for label,im in [('Original seed-123 source',D_SOURCE)]+[(str(seed),Image.open(W_OUT/W_FINALS[seed])) for seed in W_SEEDS]:
 W_AUDIT_HTML+=f'<figure><img src="{w_thumb(im,512)}"><figcaption>{html.escape(label)}</figcaption></figure>'
W_AUDIT_HTML+='</div><h2>What actually happened</h2><p>All 80 joint probes changed pixels, RGB MAE 0.001447–0.011829 against their matched unchanged probe. Every hypothesis and quality judgment was unresolved. Visible change is not established anatomical improvement.</p></body></html>'
(W_OUT/'audit.html').write_text(W_AUDIT_HTML)
display(HTML(f'<a target="_blank" href="/files/workspace/crazy_exp/{W_OUT}/audit.html">View original and all eight final images</a>'))
# Chronologically first seed, chosen before seeing endpoints: verification, not a best-image comparison.
_wverify_seed=W_SEEDS[0];_wverify_z=w_start(_wverify_seed)
_wverify_t=time.time()
with torch.no_grad():
 for i in range(50,100):
  pair,_=d_predict(_wverify_z,i)
  _wverify_z=w_step(_wverify_z,d_cfg(pair),i)
_wsaved=torch.load(W_OUT/f's{_wverify_seed}_r09_committed_step99.pt',map_location='cpu',weights_only=False)['latent_after']
W_BASELINE_VERIFY={'seed':_wverify_seed,'all_80_retained_configurations_neutral':W_ALL_NEUTRAL,'ordinary_replay_max_abs_latent_error':float((_wverify_z.cpu()-_wsaved).abs().max()),'ordinary_replay_seconds':time.time()-_wverify_t}
w_dump('ordinary_replay_verification.json',W_BASELINE_VERIFY);print('ORDINARY REPLAY VERIFICATION',W_BASELINE_VERIFY)


In [ ]:
# W14. Audit what evidence reached Jev, not only which controls were available.
print('Prediction judgments',dict(Counter(r['selection_answers']['prediction_result'] for r in W_DECISIONS)))
print('Selected uncertainty',dict(Counter(r['selection_answers']['risk'] for r in W_DECISIONS)))
_requests=sorted(W_OUT.glob('api_*_proposal_request.json'))
_responses=sorted(W_OUT.glob('api_*_proposal_response.json'))
_last_payload=json.loads(_requests[-1].read_text());_first_response=json.loads(_responses[0].read_text())
print('Actual proposal response metadata:',{k:v for k,v in _first_response.items() if k not in ['answers']})
print('Actual first hypothesis answer:',_first_response['answers']['hypothesis'])
print('Last proposal state fields:',list(_last_payload['state']))
print('History packet example:',json.dumps(_last_payload['state']['past_interventions'][-1],indent=2))
print('Full rejected-candidate records supplied to next proposal:',any('candidates' in h for h in _last_payload['state']['past_interventions']))
print('Selection reports for chronologically first round:')
for c in W_DECISIONS[0]['candidates']:
 print(c['kind'],'RGB change:',c['matched_change']['RGB_MAE'],'review:',c['visual']['report'])
for row,seeds in enumerate([W_SEEDS[2:5],W_SEEDS[5:8]],start=2):
 page='<!doctype html><html><head><meta charset="utf-8"><style>body{font:16px system-ui;background:#171a20;color:white;margin:20px}.grid{display:grid;grid-template-columns:repeat(3,1fr);gap:16px}img{width:100%}figure{margin:0}</style></head><body><h1>All results — remaining seeds</h1><div class="grid">'
 for seed in seeds:page+=f'<figure><img src="{w_thumb(Image.open(W_OUT/W_FINALS[seed]),512)}"><figcaption>{seed}</figcaption></figure>'
 page+='</div></body></html>'; (W_OUT/f'audit_row{row}.html').write_text(page)
 display(HTML(f'<a target="_blank" href="/files/workspace/crazy_exp/{W_OUT}/audit_row{row}.html">Inspect final-image row {row}</a>'))


In [ ]:
# W15. Honest conclusion and the specific controller defects exposed by this run.
W_FINDINGS='''# Completed-run review

**This run did not demonstrate a Jev-driven image-quality improvement.** All eight seeds completed all fifty scheduled continuation steps: 80 decision rounds, 160 Jev calls, no execution failures. Every retained parameter configuration was neutral. A fresh ordinary-diffusion replay of the chronologically first seed reproduced its final latent exactly (maximum absolute difference 0.0).

## What was actually tested

Jev proposed changed joint configurations in all 80 rounds. Every joint probe changed pixels relative to its same-checkpoint unchanged probe (RGB MAE 0.001447–0.011829). Therefore the executor was not ignoring its choices. However, Jev selected unchanged continuation 80/80 times, called the hypothesis and quality unresolved 80/80 times, and selected observer_conflict as the uncertainty 80/80 times. The final images are ordinary refinement under different re-noising seeds, not evidence of benefit from the rejected Jev edits.

Only six distinct joint parameter vectors appeared. FreeU interpolation was always 0.5. In 76/80 proposals all eight identity-perturbation strengths were 0.5. Layer stayed 8x8, temperature 1, CFG 7.5, all head gates 1, and commit mask full. Occasional self-guidance changes and two head-strength variations account for the remaining diversity. Available control freedom did not become state-specific control.

## Feedback and prompting limitations found in the implementation

1. **Incomplete intervention memory.** Requests included only the last three selected-branch summaries. Full rejected-branch configurations, visual observations, and measured effects were saved on disk but not supplied to the next proposal. Thus repetition could not be diagnosed well from the history Jev actually saw.
2. **Hypothesis did not condition proposal parameters.** The hypothesis and individual control questions were answered independently in one API call. Numerical edits were combined jointly, but their choices did not receive the selected semantic hypothesis. This is not the coordinated hypothesis-to-intervention process we intended.
3. **Visual descriptions lacked the needed relationships.** Qwen mainly described an open hand, sleeve, skin tone and background; it did not consistently distinguish the candidates' finger connections or local changes. Views were whole-image and reviewed separately, with no enlarged contextual views or explicit paired comparison. Its source description even said the thumb pointed upward, whereas the source thumbnail shows it extending right near the bottom of the hand. Later descriptions omitted visible abnormalities. These are uncalibrated observations.
4. **Short probes and a conservative selection rule.** Two-step clean forecasts may not reveal eventual structural consequences, and the prompt explicitly defaults to continuing when quality is unresolved. That choice was allowed and repeatedly made. This run does not isolate whether the dominant cause was the observer, choice framing, inadequate history, proposal quality, or probe horizon.
5. **Limited head-to-meaning evidence.** Per-head norms, entropy and local transformations were exposed, but there was no validated mapping from head effects to anatomical relationships. Those statistics establish mechanism and sensitivity, not a finger control.

## What the visible results support

Across the full grid, ordinary diffusion changed finger contours, palm creases, sleeve texture and sometimes jewelry. Several images retain questionable proportions or connections; the final seed has a notably reduced/unclear thumb region relative to the source. No final image is presented as a Jev success, and no last-attempt or best-seed selection was used. The original plus all eight outcomes and every rejected probe remain available.

## Cost

The recorded run took 23.32 minutes. Reported visual-observer time was about 15.24 minutes (includes its initial source check); Jev API time was about 1.38 minutes. Instrumentation counters include preflight work: 1,455 full forwards and 61 partial gradient forwards/backwards. This was not a speed improvement; wall time is not a fair uninstrumented benchmark.

## Next change worth testing

Keep this completed run immutable. For a subsequent run, supply a compact ledger of **all** tested configurations and their observed effects, including rejected ones. First obtain a supported anatomical hypothesis from full-image plus enlarged-context/paired observations; then include that explicit hypothesis in the control-selection request. Ask what a new probe would distinguish from prior probes, and measure both immediate response and a longer continuation before committing. Keep an honest unresolved option, but detect repeated non-informative probes rather than spending the remaining budget reproducing them. Validate the reviewer on the retained images before another long run. These changes address this implementation; they do not guarantee a positive result or justify forcing acceptance.
'''
(W_OUT/'review.md').write_text(W_FINDINGS)
display(Markdown(W_FINDINGS))
# Make the human-readable audit part of the existing all-image report.
_audit_text=(W_OUT/'audit.html').read_text()
_audit_text=_audit_text.replace('</body>', '<h2>Detailed audit</h2><pre>'+html.escape(W_FINDINGS)+'</pre></body>')
(W_OUT/'audit.html').write_text(_audit_text)
w_dump('audit_summary.json',{'complete':True,'seeds':8,'rounds':80,'Jev_calls':160,'selected_continue':80,'quality_unresolved':80,'risk_observer_conflict':80,'unique_joint_vectors':6,'changed_joint_probes':80,'all_committed_parameters_neutral':W_ALL_NEUTRAL,'baseline_replay':W_BASELINE_VERIFY})
print('Saved audit, full image gallery, exact replay verification, and decision summaries to',W_OUT)
